# SQL and databases

*Module 3 Part 1. Covers the deck's 67 slides.*

Everything so far has started the same way: `pd.read_csv(...)`. Somebody
handed you a file. This module is about where that file comes from in
real work, and why it is almost never a file.

## How this notebook is built

**Every query in here runs.** Not a screenshot, not a formatted example
— a real database, created in front of you, queried live. When the deck
and the database disagree, you will watch the database win.

We use **SQLite**, which is part of Python's standard library:

- Nothing to install. `import sqlite3` works right now.
- Nothing to download. Everything is built from data already in `data/`.
- It is a real SQL engine, not a toy. It is the most widely deployed
  database in the world — it is in your phone, your browser, and most
  desktop applications.

Where SQLite differs from a server database like PostgreSQL or SQL
Server, this notebook says so **at the point where it matters**, because
those differences are exactly the things that will surprise you later.

## Map

| § | Slides | What |
|---|---|---|
| 1 | 4–6 | What a database is, and why not a CSV |
| 2 | 7, 10–13 | Tables, keys, and the relational idea |
| 3 | 8–9 | Transactions and ACID — *shown, not asserted* |
| 4 | 14–17 | SQL: SELECT, WHERE, ORDER BY, LIMIT |
| 5 | 19, 35 | Constraints, and what SQLite quietly does not enforce |
| 6 | 18 | Indices — and a correction to the slide's definition |
| 7 | 20–23 | Joins, all five, with real row counts |
| 8 | 29–30 | Aggregation, GROUP BY, and NULL |
| 9 | 31 | Window functions |
| 10 | 33–37 | SQL from Python, safely |
| 11 | 41–58 | NoSQL: document and graph |
| 12 | 63–65 | Normalisation |

## 0. Setup  *(run this first)*

Two things happen here. We check which SQLite we have — the version
matters later, because some join types were only added in 2022 — and we
write one small helper so that every query in this notebook comes back
as a `DataFrame` you can read.

In [1]:
import os
import sqlite3
import textwrap

import numpy as np
import pandas as pd

print("SQLite library version:", sqlite3.sqlite_version)

# RIGHT JOIN and FULL OUTER JOIN were added to SQLite in version 3.39.
# We check rather than assume, because §7 depends on it.
version_numbers = sqlite3.sqlite_version_info
HAS_RIGHT_AND_FULL_JOIN = version_numbers >= (3, 39, 0)

if HAS_RIGHT_AND_FULL_JOIN:
    print("This version supports RIGHT JOIN and FULL OUTER JOIN.")
else:
    print("This version is older than 3.39 - §7 will skip two join types.")


def find_data(filename):
    """Return the first path that exists, so this runs from either folder."""
    candidates = [filename, os.path.join("data", filename)]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        "could not find {} - looked in {}".format(filename, candidates)
    )

SQLite library version: 3.53.4
This version supports RIGHT JOIN and FULL OUTER JOIN.


In [2]:
# One database, held in memory, for the whole notebook.
# ":memory:" means nothing is written to disk - close Python and it is gone.
db = sqlite3.connect(":memory:")


def run(sql, params=()):
    """Run a SELECT and return the result as a DataFrame."""
    return pd.read_sql_query(sql, db, params=params)


def execute(sql, params=()):
    """Run a statement that changes something (CREATE, INSERT, UPDATE...)."""
    cursor = db.execute(sql, params)
    db.commit()
    return cursor


def show(sql, params=()):
    """Print the SQL, then run it and display the result. Used throughout."""
    print(textwrap.dedent(sql).strip())
    print("-" * 60)
    return run(sql, params)


print("Connected. Nothing on disk - this database lives in RAM.")

Connected. Nothing on disk - this database lives in RAM.


---

## 1. What a database is, and why not a CSV  *(slides 4–6)*

Slide 4 defines a database as *"a computer system that manages storage
and querying of data"*, and slide 5 says it is *"more robust than text,
CSV or JSON files"*.

That word **robust** is doing a lot of work, and the deck does not
unpack it. Let's make it concrete, because "robust" is the entire reason
this module exists.

### The question

You have a table of employees. Somebody must not be able to record a
salary of `"banana"`. How do you stop them?

**In a CSV, you cannot.** A CSV has no idea what a salary is. It is
text, separated by commas. Watch.

In [5]:
# A CSV is just text. It will hold absolutely anything you type.
csv_text = "name,salary\n"
csv_text += "Alice,50000\n"
csv_text += "Bob,banana\n"          # <- nonsense, and nothing objects
csv_text += "Carol,-9999999\n"      # <- a negative salary, also fine

from io import StringIO
employees_csv = pd.read_csv(StringIO(csv_text))

print(employees_csv)
print()
print("dtype of the salary column:", employees_csv["salary"].dtype)
print()
print("pandas gave up and called the whole column 'object' (text),")
print("because one bad value poisons the entire column.")

    name    salary
0  Alice     50000
1    Bob    banana
2  Carol  -9999999

dtype of the salary column: object

pandas gave up and called the whole column 'object' (text),
because one bad value poisons the entire column.


Notice what actually went wrong. It is not just that `banana` is in
there. It is that **`Alice`'s perfectly good salary of 50000 is now text
too**. One bad row silently changed the type of the whole column, and
every sum, mean and comparison you write downstream is now wrong or
broken.

Now the same thing with a database. We declare what a salary *is*, and
we declare what makes it valid.

> **Predict first.** We are about to tell the database that `salary` must be an integer and must not be negative, then try to insert `'banana'` and `-9999999`. What happens to each?
>
> Put your answer in the chat before we run it.

In [3]:
execute("""
CREATE TABLE employees (
    id      INTEGER PRIMARY KEY,
    name    TEXT    NOT NULL,
    salary  INTEGER NOT NULL CHECK (salary >= 0)
)
""")

# The good row goes in without complaint.
execute("INSERT INTO employees (name, salary) VALUES (?, ?)", ("Alice", 50000))
print("Alice inserted.")

# The negative salary is refused by the CHECK constraint.
try:
    execute("INSERT INTO employees (name, salary) VALUES (?, ?)",
            ("Carol", -9999999))
except sqlite3.IntegrityError as problem:
    print("Carol REFUSED ->", problem)

# A missing name is refused by NOT NULL.
try:
    execute("INSERT INTO employees (name, salary) VALUES (?, ?)", (None, 1000))
except sqlite3.IntegrityError as problem:
    print("Nameless REFUSED ->", problem)

print()
print(run("SELECT * FROM employees"))

Alice inserted.
Carol REFUSED -> CHECK constraint failed: salary >= 0
Nameless REFUSED -> NOT NULL constraint failed: employees.name

   id   name  salary
0   1  Alice   50000


**That is what "robust" means.** The rules live *with the data*, in one
place, and they are enforced against every program that ever writes to
it — your notebook, your colleague's script, the web application, the
person poking at it by hand at 2am.

In a CSV, the rules live in the head of whoever wrote the loading code,
and only while they remember them.

### One honest caveat, immediately

You may have noticed we did not try to insert `'banana'`. There is a
reason, and it is a genuine SQLite quirk that §5 covers properly:
**SQLite would have accepted it.** A CHECK constraint catches the
negative number, but SQLite's type declarations are advisory, not
enforced. PostgreSQL and SQL Server would refuse it outright.

We will look straight at that in §5 rather than pretend otherwise.

### The rest of slide 5's list, briefly

- **A central source of truth.** One copy that many programs read, not
  fourteen slightly different spreadsheets on fourteen laptops.
- **Concurrency.** Two people writing to the same CSV at the same
  moment produce a corrupt file. A database is built for this — that is
  §3.
- **Retrieval without loading everything.** `pd.read_csv` on a 200 GB
  file loads 200 GB. A database with an index reads the handful of
  pages it needs — that is §6.

---

## 2. Tables, keys, and the relational idea  *(slides 7, 10–13)*

Slide 7 lists the elements of a database — tables, keys, queries and
views, functions and procedures, types, triggers, jobs. Slide 10 gives
the organising rule in one sentence:

> **Each table in a database is devoted to one domain.**

Slides 11 and 21 illustrate it with a music library: `Artists`,
`Recordings`, `Genre`. We are going to build exactly that, because in
§7 we run the deck's own query against it.

### Primary and foreign keys  *(slide 19)*

| | |
|---|---|
| **Primary key** | uniquely identifies each row in *this* table. One per table. |
| **Foreign key** | points at a primary key in *another* table. Any number per table. |

Slide 11's example: `ArtistID` is the **primary** key in `Artists`, and
a **foreign** key in `Recordings`.

### Why bother? The duplication argument

Slide 19 notes that foreign keys "reduce storage and make database
maintenance much easier". That undersells it. The real argument is
about **being wrong in only one place**.

If every recording stored its artist's name as text, then an artist who
changes their name requires you to find and update every row — and if
you miss one, your database now disagrees with itself. Store the name
once, point at it with an ID, and there is no way to be inconsistent.

In [4]:
# Slide 11's schema, built for real.
execute("""
CREATE TABLE Artists (
    ArtistID   INTEGER PRIMARY KEY,
    ArtistName TEXT NOT NULL
)
""")

execute("""
CREATE TABLE Genre (
    GenreID INTEGER PRIMARY KEY,
    Genre   TEXT NOT NULL
)
""")

execute("""
CREATE TABLE Recordings (
    RecordingID INTEGER PRIMARY KEY,
    Title       TEXT NOT NULL,
    AlbumCover  TEXT,
    ArtistID    INTEGER REFERENCES Artists(ArtistID),   -- foreign key
    GenreID     INTEGER REFERENCES Genre(GenreID)       -- foreign key
)
""")

artists = [(1, "Fleetwood Mac"), (2, "Miles Davis"),
           (3, "Nina Simone"), (4, "Led Zeppelin")]
db.executemany("INSERT INTO Artists VALUES (?, ?)", artists)

genres = [(1, "rock"), (2, "jazz"), (3, "soul")]
db.executemany("INSERT INTO Genre VALUES (?, ?)", genres)

recordings = [
    (1, "Rumours",             "rumours.jpg",   1, 1),
    (2, "Tusk",                "tusk.jpg",      1, 1),
    (3, "Kind of Blue",        "kob.jpg",       2, 2),
    (4, "I Put a Spell on You", "spell.jpg",    3, 3),
    (5, "Led Zeppelin IV",     "lz4.jpg",       4, 1),
]
db.executemany("INSERT INTO Recordings VALUES (?, ?, ?, ?, ?)", recordings)
db.commit()

print("Artists")
print(run("SELECT * FROM Artists"))
print()
print("Recordings")
print(run("SELECT * FROM Recordings"))

Artists
   ArtistID     ArtistName
0         1  Fleetwood Mac
1         2    Miles Davis
2         3    Nina Simone
3         4   Led Zeppelin

Recordings
   RecordingID                 Title   AlbumCover  ArtistID  GenreID
0            1               Rumours  rumours.jpg         1        1
1            2                  Tusk     tusk.jpg         1        1
2            3          Kind of Blue      kob.jpg         2        2
3            4  I Put a Spell on You    spell.jpg         3        3
4            5       Led Zeppelin IV      lz4.jpg         4        1


Read the `Recordings` table and notice what is **not** in it: no artist
names, no genre names. Just numbers pointing elsewhere. `ArtistID = 1`
appears twice, and "Fleetwood Mac" is stored exactly once.

That is the relational idea in one picture.

### Schema  *(slide 13)*

Slide 13 distinguishes the **table-level** schema (columns, primary key)
from the **database-level** schema (the whole design: tables, keys,
integrity constraints).

SQLite will show you its own. This is the database describing itself —
useful whenever you meet a database you did not build.

In [5]:
schema = run("""
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""")

for _, row in schema.iterrows():
    print("=" * 60)
    print(row["sql"])

CREATE TABLE Artists (
    ArtistID   INTEGER PRIMARY KEY,
    ArtistName TEXT NOT NULL
)
CREATE TABLE Genre (
    GenreID INTEGER PRIMARY KEY,
    Genre   TEXT NOT NULL
)
CREATE TABLE Recordings (
    RecordingID INTEGER PRIMARY KEY,
    Title       TEXT NOT NULL,
    AlbumCover  TEXT,
    ArtistID    INTEGER REFERENCES Artists(ArtistID),   -- foreign key
    GenreID     INTEGER REFERENCES Genre(GenreID)       -- foreign key
)
CREATE TABLE employees (
    id      INTEGER PRIMARY KEY,
    name    TEXT    NOT NULL,
    salary  INTEGER NOT NULL CHECK (salary >= 0)
)


---

## 3. Transactions and ACID  *(slides 8–9)*

This is the most important section in the module and the one most often
learned as four words to recite in an interview. We are going to make it
happen instead.

Slide 8 gives the canonical example:

> Transfer 1,000 from savings to checking.
> **1.** Withdraw 1,000 from savings. **2.** Deposit 1,000 into checking.

The whole problem in one observation: **between step 1 and step 2, the
money does not exist anywhere.** If the program crashes in that gap, or
the power fails, or step 2 hits an error, 1,000 has evaporated.

Slide 9 names the four guarantees that stop this. Take **Atomicity**
first — *"if one part of the transaction fails, the entire transaction
fails; the database state is left unchanged ('all or nothing')"*.

Let's break a transfer on purpose.

In [6]:
execute("""
CREATE TABLE accounts (
    name    TEXT PRIMARY KEY,
    balance INTEGER NOT NULL CHECK (balance >= 0)
)
""")

db.executemany("INSERT INTO accounts VALUES (?, ?)",
               [("savings", 5000), ("checking", 200)])
db.commit()

print(run("SELECT * FROM accounts"))
print()
print("total in the bank:",
      run("SELECT SUM(balance) AS total FROM accounts")["total"][0])

       name  balance
0   savings     5000
1  checking      200

total in the bank: 5200


> **Predict first.** We will transfer 9,000 out of savings, which only holds 5,000. Step 1 (the withdrawal) will fail the `CHECK (balance >= 0)` rule. What will `checking` hold afterwards — 200, or 9,200?
>
> Put your answer in the chat before we run it.

In [7]:
def transfer(amount, source, destination):
    """Move money. Either both steps happen, or neither does."""
    try:
        # BEGIN starts the transaction: nothing below is final until COMMIT.
        db.execute("BEGIN")

        db.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?",
                   (amount, source))
        db.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?",
                   (amount, destination))

        db.execute("COMMIT")
        return "committed"

    except sqlite3.IntegrityError as problem:
        # ROLLBACK undoes every step since BEGIN, as if none had happened.
        db.execute("ROLLBACK")
        return "rolled back: {}".format(problem)


print("A transfer that cannot succeed:")
print("  ", transfer(9000, "savings", "checking"))
print()
print(run("SELECT * FROM accounts"))
print()

total = run("SELECT SUM(balance) AS total FROM accounts")["total"][0]
print("total in the bank:", total)
assert total == 5200, "money was created or destroyed - atomicity failed"
print("Unchanged. No money was created and none was lost.")

A transfer that cannot succeed:
   rolled back: CHECK constraint failed: balance >= 0

       name  balance
0   savings     5000
1  checking      200

total in the bank: 5200
Unchanged. No money was created and none was lost.


**That is atomicity, and it is not a slogan.** The first `UPDATE`
*did* run. The withdrawal happened. Then the second statement failed the
CHECK, and `ROLLBACK` reached back and undid the withdrawal too.

Run a transfer that *should* work, to see the other half.

In [8]:
print("A transfer that can succeed:")
print("  ", transfer(1000, "savings", "checking"))
print()
print(run("SELECT * FROM accounts"))

total = run("SELECT SUM(balance) AS total FROM accounts")["total"][0]
print()
print("total in the bank:", total)
assert total == 5200, "the total must never change during a transfer"
print("Money moved. The total is still", total, "- as it must be.")

A transfer that can succeed:
   committed

       name  balance
0   savings     4000
1  checking     1200

total in the bank: 5200
Money moved. The total is still 5200 - as it must be.


### The other three letters

**Consistency** — *"any transaction brings the database from one valid
state to another"*. "Valid" means *your rules*: the `CHECK`, the
`NOT NULL`, the foreign keys. In the failed transfer, the invalid state
(savings = −4,000) was never allowed to become real.

**Isolation** — *"concurrent execution of transactions results in a
system state that would have been obtained if the transactions were
executed serially"*. Two transfers running at the same instant must not
be able to read each other's half-finished work. This is the letter you
cannot demonstrate in a single-threaded notebook, and it is the one that
causes the worst production bugs. The classic failure: two people
withdraw the last 100 dollars simultaneously, both read a balance of
100, both are allowed, and the account ends at −100.

**Durability** — *"after a transaction has been committed it will be
unaffected by power loss, system crashes, or errors"*. Once `COMMIT`
returns, the data is on disk, not in a buffer that a crash would lose.

> Our database is in memory (`:memory:`), so we have **atomicity,
> consistency and isolation, but not durability.** Close Python and it
> is gone. That is a property of our choice today, not of SQLite — a
> file-backed SQLite database is durable.

---

## 4. SQL: the basics  *(slides 14–17)*

Slide 14 makes an important point that is easy to skim past: SQL is
**declarative**.

> You describe the rowset you want. The database decides **how** to get
> it.

In pandas you write the *steps*: filter, then sort, then take the head.
In SQL you write the *destination*, and the query planner picks the
route. You will see it pick one in §6.

### The three divisions  *(slide 14)*

| Division | What it does | Statements |
|---|---|---|
| **DDL** — data definition | builds the shape | `CREATE`, `ALTER`, `DROP` |
| **DML** — data manipulation | moves the contents | `SELECT`, `INSERT`, `UPDATE`, `DELETE` |
| **DCL** — data control | who is allowed | `GRANT`, `REVOKE` |

We have already used DDL (`CREATE TABLE`) and DML (`INSERT`). DCL only
appears on a server database with real users — SQLite has no user
accounts at all, because its access control is the file's permissions.

### Now some real data

Time to leave toy tables. We load the bikeshare data you know from
Part 1 straight into a table.

In [9]:
day = pd.read_csv(find_data("bikeshare-day.csv"), parse_dates=["dteday"])
hour = pd.read_csv(find_data("bikeshare-hour.csv"), parse_dates=["dteday"])

# pandas will create the table and insert every row for us.
day.to_sql("day", db, index=False, if_exists="replace")
hour.to_sql("hour", db, index=False, if_exists="replace")

print("day  rows:", run("SELECT COUNT(*) AS n FROM day")["n"][0])
print("hour rows:", run("SELECT COUNT(*) AS n FROM hour")["n"][0])

# Sanity check: the two tables must agree on total rides.
totals = run("""
    SELECT (SELECT SUM(cnt) FROM day)  AS from_day,
           (SELECT SUM(cnt) FROM hour) AS from_hour
""")
print()
print(totals)
assert totals["from_day"][0] == totals["from_hour"][0], "tables disagree"
print("Both tables agree. Good.")

day  rows: 731
hour rows: 17379

   from_day  from_hour
0   3292679    3292679
Both tables agree. Good.


### Slide 15's shape

Slide 15 gives the pattern:

```sql
SELECT columns FROM table WHERE condition ORDER BY column LIMIT n
```

Read in that written order it is confusing, because that is **not the
order the database evaluates it in.** The engine works like this:

| Order | Clause | Meaning |
|---|---|---|
| 1 | `FROM` | which table |
| 2 | `WHERE` | throw away rows that fail the test |
| 3 | `GROUP BY` | gather survivors into buckets *(§8)* |
| 4 | `HAVING` | throw away whole buckets *(§8)* |
| 5 | `SELECT` | choose and compute columns |
| 6 | `ORDER BY` | sort what is left |
| 7 | `LIMIT` | keep the first n |

**`WHERE` before `SELECT`** is the one to remember. It explains a whole
family of errors you are about to meet in §8.

In [10]:
show("""
    SELECT dteday, cnt, temp
    FROM day
    WHERE cnt > 8000
    ORDER BY cnt DESC
    LIMIT 5
""")

SELECT dteday, cnt, temp
FROM day
WHERE cnt > 8000
ORDER BY cnt DESC
LIMIT 5
------------------------------------------------------------


,dteday,cnt,temp
0,2012-09-15 00:00:00,8714,0.608333
1,2012-09-29 00:00:00,8555,0.542500
2,2012-09-22 00:00:00,8395,0.650000
3,2012-03-23 00:00:00,8362,0.601667
4,2012-05-19 00:00:00,8294,0.600000


### Two portability notes the deck raises  *(slides 16–17)*

**Slide 17: "SQL is case-insensitive."** True of *keywords* and, in
most engines, of identifiers. It is **not** true of string data.

In [ ]:
print("Keywords - both work, identical result:")
print(run("select count(*) as n from day where yr = 1")["n"][0])
print(run("SELECT COUNT(*) AS n FROM day WHERE yr = 1")["n"][0])
print()

print("String VALUES are a different matter:")
print(show("""
    SELECT ArtistName
    FROM Artists
    WHERE ArtistName = 'miles davis'
"""))
print("-> no rows. The stored value is 'Miles Davis'.")
print()
print(show("""
    SELECT ArtistName
    FROM Artists
    WHERE LOWER(ArtistName) = 'miles davis'
"""))

**Slide 15 uses `LIMIT`, and that is not universal.** Slide 17 warns
that vendors extend the standard; this is a case where they disagree on
something basic.

| Engine | Syntax |
|---|---|
| SQLite, PostgreSQL, MySQL | `SELECT ... ORDER BY x LIMIT 10` |
| SQL Server | `SELECT TOP 10 ... ORDER BY x` |
| Oracle (older) | `WHERE ROWNUM <= 10` |
| ANSI standard | `ORDER BY x FETCH FIRST 10 ROWS ONLY` |

The deck talks about SQL Server on slides 16, 18 and 26 but writes
`LIMIT` on slide 15. Both are fine; they are simply different dialects.
**Know which engine you are typing at.**

**Slide 16: identifiers with spaces** need delimiters — `[my table]` in
SQL Server, `"my table"` in SQLite and the standard, backticks in MySQL.
The better advice is to never put a space in a column name.

---

## 5. What SQLite does *not* enforce  *(slides 19, 35)*

Slide 35 shows this SQLite table:

```sql
CREATE TABLE employee (
    staff_number INTEGER PRIMARY KEY,
    fname VARCHAR(20),
    lname VARCHAR(30),
    date_joined DATE);
```

It looks like it promises three things: names of at most 20 and 30
characters, and a real date. **It promises none of them.** This is not
a criticism of the slide's syntax — the statement is valid and runs. It
is a fact about SQLite that will bite you, so let's watch it.

> **Predict first.** We create the slide's exact table, then insert a 500-character first name and the text `'not-a-date'` into the DATE column. How many of those two are rejected?
>
> Put your answer in the chat before we run it.

In [ ]:
execute("""
CREATE TABLE employee (
    staff_number INTEGER PRIMARY KEY,
    fname VARCHAR(20),
    lname VARCHAR(30),
    date_joined DATE)
""")

execute("INSERT INTO employee VALUES (?, ?, ?, ?)",
        (1, "x" * 500, "Smith", "not-a-date"))

result = run("""
    SELECT staff_number,
           LENGTH(fname)      AS fname_length,
           date_joined,
           TYPEOF(fname)      AS fname_type,
           TYPEOF(date_joined) AS date_type
    FROM employee
""")
print(result)
print()
print("Neither was rejected.")
print("VARCHAR(20) holds", result["fname_length"][0], "characters.")
print("The DATE column holds the text", repr(result["date_joined"][0]))

### Why: type affinity

SQLite does not have static column types. It has **type affinity** — a
column has a *preference*, and SQLite converts when it easily can and
shrugs when it cannot.

- `VARCHAR(20)` gives TEXT affinity. **The 20 is parsed and discarded.**
- `DATE` is not a SQLite type at all. There is no date type. Dates are
  stored as TEXT (`'2026-09-09'`), INTEGER (Unix seconds) or REAL
  (Julian day), and it is your job to be consistent.

Affinity is not *nothing*, though — watch it convert when it can.

In [ ]:
execute("CREATE TABLE affinity_demo (n INTEGER)")

execute("INSERT INTO affinity_demo VALUES (?)", ("42",))     # numeric text
execute("INSERT INTO affinity_demo VALUES (?)", ("banana",))  # not numeric

print(run("SELECT n, TYPEOF(n) AS stored_as FROM affinity_demo"))
print()
print("'42' was converted to an integer - affinity did its job.")
print("'banana' could not be, so it was stored as text in an INTEGER column.")
print()
print("PostgreSQL and SQL Server would have refused 'banana' outright.")

> **This is the single biggest difference between SQLite and a server
> database.** Prototype in SQLite, and a `VARCHAR(20)` that silently
> accepted 500 characters will fail loudly on the day you move to
> PostgreSQL. Enforce what matters with `CHECK`, as §1 did — those *are*
> enforced.

### The foreign key that is not a foreign key

There is a second, sharper surprise, and it undercuts everything slide
19 says about referential integrity. In §2 we declared
`ArtistID INTEGER REFERENCES Artists(ArtistID)`.

**SQLite ignores it by default.**

> **Predict first.** `Artists` has four rows, with IDs 1 to 4. We are about to insert a recording with `ArtistID = 999`, which does not exist. Does the foreign key stop it?
>
> Put your answer in the chat before we run it.

In [ ]:
print("foreign key enforcement is currently:",
      db.execute("PRAGMA foreign_keys").fetchone()[0], "(0 means OFF)")
print()

execute("INSERT INTO Recordings VALUES (?, ?, ?, ?, ?)",
        (99, "Album By Nobody", "ghost.jpg", 999, 1))

orphans = run("""
    SELECT r.RecordingID, r.Title, r.ArtistID
    FROM Recordings r
    LEFT JOIN Artists a ON r.ArtistID = a.ArtistID
    WHERE a.ArtistID IS NULL
""")
print("Rows pointing at an artist that does not exist:")
print(orphans)
print()
print("The database accepted a recording by an artist who does not exist.")

In [ ]:
# Clean up the orphan, then turn enforcement on and try again.
execute("DELETE FROM Recordings WHERE RecordingID = 99")

db.execute("PRAGMA foreign_keys = ON")
print("foreign key enforcement is now:",
      db.execute("PRAGMA foreign_keys").fetchone()[0], "(1 means ON)")
print()

try:
    execute("INSERT INTO Recordings VALUES (?, ?, ?, ?, ?)",
            (99, "Album By Nobody", "ghost.jpg", 999, 1))
    print("still accepted - unexpected")
except sqlite3.IntegrityError as problem:
    print("REFUSED ->", problem)

remaining = run("SELECT COUNT(*) AS n FROM Recordings")["n"][0]
print()
print("Recordings still holds", remaining, "rows.")
assert remaining == 5, "the orphan row should not exist"

**`PRAGMA foreign_keys = ON` is per-connection and off by default.**
It is off for backwards compatibility with databases written before
SQLite supported it. If you build a schema in SQLite and rely on
`REFERENCES` without that pragma, you have written documentation, not a
constraint.

---

## 6. Indices  *(slide 18)*

Slide 18 opens with a definition that needs correcting before we go on:

> *"An index is a column of unique numbers (one per row) used to speed
> up query performance."*

**An index is not a column, and it is not necessarily unique.** Three
corrections, and they matter:

1. **It is a separate data structure**, not a column in your table.
   Typically a B-tree, held beside the table. Your table's columns are
   unchanged when you add one.
2. **It does not have to be unique.** An index on "suburb" in a
   customer table has thousands of rows per value. `CREATE UNIQUE INDEX`
   is the *special* case, not the definition.
3. **It stores the indexed values, sorted, with pointers to the rows.**
   That is *why* it is fast: sorted data can be binary-searched instead
   of scanned.

A better one-line definition: **an index is a sorted lookup structure
that lets the engine find matching rows without reading every row.**

The rest of the slide is right, and the last bullet is genuinely good
professional advice: for a large bulk insert, drop the index, insert,
then rebuild — maintaining it row by row during the insert is slower.

### Watch the planner change its mind

`EXPLAIN QUERY PLAN` shows the route the engine chose. This is the
declarative promise from §4 made visible.

In [ ]:
# A table big enough for the difference to be real: 17,379 rows.
execute("CREATE TABLE hour_indexed AS SELECT * FROM hour")

print("BEFORE any index:")
print(run("EXPLAIN QUERY PLAN SELECT * FROM hour_indexed WHERE dteday = '2012-09-15'"))

In [ ]:
execute("CREATE INDEX idx_hour_date ON hour_indexed (dteday)")

print("AFTER creating an index on dteday:")
print(run("EXPLAIN QUERY PLAN SELECT * FROM hour_indexed WHERE dteday = '2012-09-15'"))

Read the `detail` column in both.

- **`SCAN hour_indexed`** — read all 17,379 rows and test each one.
- **`SEARCH hour_indexed USING INDEX idx_hour_date`** — go straight to
  the matching rows.

Nothing about the query changed. We changed what was *available*, and
the planner chose differently. That is the declarative idea paying off.

Now time it, because "faster" should be a measurement.

In [ ]:
import time

def time_query(sql, repeats=200):
    """Run a query many times and return the average milliseconds."""
    start = time.perf_counter()
    for _ in range(repeats):
        db.execute(sql).fetchall()
    elapsed = time.perf_counter() - start
    return (elapsed / repeats) * 1000


query = "SELECT * FROM {} WHERE dteday = '2012-09-15'"

with_index = time_query(query.format("hour_indexed"))
without_index = time_query(query.format("hour"))     # never indexed

print("no index : {:.3f} ms per query".format(without_index))
print("index    : {:.3f} ms per query".format(with_index))
print("speed-up : {:.1f}x".format(without_index / with_index))

> **That speed-up is large even here**, on a table of only 17,379 rows,
> because the query is *selective* — it wants one day out of two years,
> so the index skips almost everything while the scan reads it all.
>
> The gap widens with table size: a scan costs O(n), an index lookup
> roughly O(log n). It also **narrows to nothing, or goes negative, when
> a query is not selective.** Asking for half the table is faster by
> plain scan, and a good planner will ignore your index and scan anyway.
> An index helps you find a needle, not harvest a field.

**Indexes are not free.** They take disk space, and every `INSERT`,
`UPDATE` and `DELETE` must update every index on the table. Index the
columns you filter and join on. Do not index everything.

---

## 7. Joins  *(slides 20–23)*

Slide 20 says a join is *"analogous to a set operation in
mathematics"*, which is the right way in. Slides 22 and 23 give the
types. **Slide 22 is correct. Slide 23 is not**, and we will run it to
find out how.

First, build slide 22's exact example data. The slide shows two tables
of makers:

| Table 1 | Table 2 |
|---|---|
| aaa | aaa |
| bbb | xxx |
| ccc | ccc |
| ddd | yyy |
| eee | eee |
| fff | fff |

Four makers appear in both (aaa, ccc, eee, fff). One table has bbb and
ddd alone; the other has xxx and yyy alone. That structure is the whole
lesson.

In [11]:
execute("CREATE TABLE cars (car TEXT, maker TEXT)")
execute("CREATE TABLE trucks (truck TEXT, maker TEXT)")

car_rows = [("car-aaa", "aaa"), ("car-bbb", "bbb"), ("car-ccc", "ccc"),
            ("car-ddd", "ddd"), ("car-eee", "eee"), ("car-fff", "fff")]
truck_rows = [("truck-aaa", "aaa"), ("truck-xxx", "xxx"), ("truck-ccc", "ccc"),
              ("truck-yyy", "yyy"), ("truck-eee", "eee"), ("truck-fff", "fff")]

db.executemany("INSERT INTO cars VALUES (?, ?)", car_rows)
db.executemany("INSERT INTO trucks VALUES (?, ?)", truck_rows)
db.commit()

print("cars  :", [r[1] for r in car_rows])
print("trucks:", [r[1] for r in truck_rows])
print("shared:", sorted(set(r[1] for r in car_rows) &
                        set(r[1] for r in truck_rows)))

cars  : ['aaa', 'bbb', 'ccc', 'ddd', 'eee', 'fff']
trucks: ['aaa', 'xxx', 'ccc', 'yyy', 'eee', 'fff']
shared: ['aaa', 'ccc', 'eee', 'fff']


> **Predict first.** Six rows in each table, four makers in common. For each of INNER, LEFT, RIGHT, FULL OUTER and CROSS — how many rows come back? Write down five numbers.
>
> Put your answer in the chat before we run it.

In [ ]:
inner = run("""
    SELECT cars.car, trucks.truck
    FROM cars
    INNER JOIN trucks ON cars.maker = trucks.maker
""")
print("INNER JOIN -", len(inner), "rows: only makers present in BOTH")
print(inner)

In [ ]:
left = run("""
    SELECT cars.car, trucks.truck
    FROM cars
    LEFT JOIN trucks ON cars.maker = trucks.maker
""")
print("LEFT JOIN -", len(left), "rows: every car, matched where possible")
print(left)
print()
print("bbb and ddd have no truck, so truck is None (SQL NULL) - exactly")
print("what slide 22 says: 'unmatched rows will have NULL in place of truck'.")

In [ ]:
if HAS_RIGHT_AND_FULL_JOIN:
    right = run("""
        SELECT cars.car, trucks.truck
        FROM cars
        RIGHT JOIN trucks ON cars.maker = trucks.maker
    """)
    print("RIGHT JOIN -", len(right), "rows: every truck, matched where possible")
    print(right)
else:
    print("This SQLite is older than 3.39 - RIGHT JOIN unavailable.")
    print("Equivalent: swap the tables and use LEFT JOIN.")

In [ ]:
if HAS_RIGHT_AND_FULL_JOIN:
    full = run("""
        SELECT cars.car, trucks.truck
        FROM cars
        FULL OUTER JOIN trucks ON cars.maker = trucks.maker
    """)
    print("FULL OUTER JOIN -", len(full), "rows: everything from both sides")
    print(full)
    print()
    print("4 matched + 2 cars with no truck + 2 trucks with no car = 8")
else:
    print("This SQLite is older than 3.39 - FULL OUTER JOIN unavailable.")

In [ ]:
cross = run("""
    SELECT cars.car, trucks.truck
    FROM cars
    CROSS JOIN trucks
""")
print("CROSS JOIN -", len(cross), "rows: every possible pairing, 6 x 6")
print(cross.head(8))
print("... and so on to", len(cross))

assert len(inner) == 4
assert len(left) == 6
assert len(cross) == 36
if HAS_RIGHT_AND_FULL_JOIN:
    assert len(right) == 6
    assert len(full) == 8
print()
print("All row counts confirmed.")

### Now slide 23

Here is what slide 23 says, in full:

> **Outer Join**
> - All rows from 1st table, matched with all rows from 2nd table:
>   `SELECT cars.car, trucks.truck FROM cars OUTER JOIN trucks ON cars.maker = trucks.maker`
> - Returns every possible pairing of car and truck
> - Uses: creating contingency tables, creating dummy data for testing

**Three different operations are on that one slide**, and they
contradict each other:

| Element on the slide | What it describes |
|---|---|
| The title, "Outer Join", and the Venn diagram | **FULL OUTER JOIN** — 8 rows |
| "every possible pairing", contingency tables, dummy data | **CROSS JOIN** — 36 rows |
| The code, with `ON cars.maker = trucks.maker` | neither — an `ON` clause contradicts "every pairing" |

The two operations are not close. One gives 8 rows, the other 36.

And the code itself does not run. Watch.

In [ ]:
# Slide 23's code, exactly as printed. Read the error.
run("""
    SELECT cars.car, trucks.truck
    FROM cars
    OUTER JOIN trucks ON cars.maker = trucks.maker
""")

`unknown join type: outer`.

**Bare `OUTER JOIN` is not valid SQL.** The word `OUTER` is an optional
modifier on `LEFT`, `RIGHT` or `FULL` — you write `LEFT OUTER JOIN`
(same as `LEFT JOIN`), `FULL OUTER JOIN`, and so on. On its own it names
nothing, which is why the engine cannot parse it.

### What the slide should say

The two things it merged:

**CROSS JOIN** — every pairing, no `ON` clause. Rows = *n* × *m*. This
is what "every possible pairing of car and truck", contingency tables
and dummy data all describe. 36 rows.

**FULL OUTER JOIN** — matched rows, plus unmatched rows from *both*
sides padded with NULL. This is what the title and Venn diagram
describe. 8 rows.

We ran both above. Here they are side by side so the gap is not
abstract.

In [ ]:
print("CROSS JOIN      :", len(cross), "rows  <- 'every possible pairing'")
if HAS_RIGHT_AND_FULL_JOIN:
    print("FULL OUTER JOIN :", len(full), "rows  <- the title and the Venn diagram")
    print()
    print("Ratio: {:.1f}x. These are not the same operation.".format(
        len(cross) / len(full)))

> **One caution about CROSS JOIN that the deck does not give.** Row
> counts multiply. Two tables of 10,000 rows cross-joined produce
> **100 million** rows. This is the classic way to accidentally hang a
> production database — usually by writing `FROM a, b` and forgetting
> the `WHERE` that was meant to join them. If a query is taking
> forever, an accidental cross join is the first thing to check.

### Slide 21's compound join, run for real

Slide 21 asks: *return artist name and album cover for every album of
the 'rock' genre*. It gives a three-table join. This one is correct —
here it is against the tables we built in §2.

In [ ]:
show("""
    SELECT Artists.ArtistName, Recordings.AlbumCover
    FROM Artists
    INNER JOIN Recordings ON Artists.ArtistID = Recordings.ArtistID
    INNER JOIN Genre      ON Recordings.GenreID = Genre.GenreID
    WHERE Genre.Genre = 'rock'
""")

Three tables, two `ON` conditions. The rule generalises: joining *n*
tables needs *n − 1* join conditions. Forget one and you have silently
written a cross join — which is slide 20's warning that *"compound joins
involving many tables can be very slow"*, and now you know the mechanism
behind it.

---

## 8. Aggregation and GROUP BY  *(slides 29–30)*

Slide 29 lists `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` and describes COUNT
as *"counts all rows meeting criterion"*.

That is true of `COUNT(*)` and **false of `COUNT(column)`**, and the
difference is where NULL enters. NULL is the single most common source
of quietly wrong SQL, so we start there.

In [ ]:
execute("CREATE TABLE survey (person TEXT, score INTEGER)")
db.executemany("INSERT INTO survey VALUES (?, ?)",
               [("a", 1), ("b", 2), ("c", None), ("d", 2)])
db.commit()

print(run("SELECT * FROM survey"))
print()
print(show("""
    SELECT COUNT(*)              AS count_star,
           COUNT(score)          AS count_score,
           COUNT(DISTINCT score) AS count_distinct,
           SUM(score)            AS sum_score,
           AVG(score)            AS avg_score
    FROM survey
"""))

Read those five numbers against the four rows:

| Result | Value | Why |
|---|---|---|
| `COUNT(*)` | 4 | counts **rows** |
| `COUNT(score)` | 3 | counts **non-NULL values** — c is skipped |
| `COUNT(DISTINCT score)` | 2 | the distinct non-NULL values are 1 and 2 |
| `SUM(score)` | 5 | 1 + 2 + 2, NULL ignored |
| `AVG(score)` | 1.667 | **5 ÷ 3, not 5 ÷ 4** |

**That `AVG` is the one to remember.** It divided by 3, the number of
non-NULL scores, not by 4, the number of people. If you wanted "average
score per person, counting non-responses as zero", SQL did not give it
to you and did not warn you.

In [ ]:
print(show("""
    SELECT AVG(score)                     AS avg_ignoring_nulls,
           AVG(COALESCE(score, 0))        AS avg_nulls_as_zero,
           SUM(score) * 1.0 / COUNT(*)    AS same_thing_by_hand
    FROM survey
"""))
print("COALESCE(score, 0) replaces NULL with 0 before averaging.")
print("Which one is correct depends on your question, not on SQL.")

### GROUP BY  *(slide 30)*

Slide 30: *"grouping allows aggregation functions to be applied to
subsets based on row-level criteria"*. Its example is
`SELECT agent_name, SUM(sales) ... GROUP BY agent_name`.

The mental model: **`GROUP BY` sorts rows into buckets; the aggregate
function collapses each bucket to one row.**

In [ ]:
show("""
    SELECT weathersit,
           COUNT(*)      AS days,
           SUM(cnt)      AS total_rides,
           ROUND(AVG(cnt), 1) AS avg_rides,
           MIN(cnt)      AS quietest,
           MAX(cnt)      AS busiest
    FROM day
    GROUP BY weathersit
    ORDER BY weathersit
""")

### The error everyone makes once: WHERE against HAVING

Recall the evaluation order from §4: **`WHERE` runs before grouping,
`HAVING` runs after.** So `WHERE` cannot see an aggregate — at the
moment `WHERE` runs, the groups do not exist yet.

> **Predict first.** We filter for `WHERE SUM(cnt) > 1000000`. Does it return nothing, or does it fail?
>
> Put your answer in the chat before we run it.

In [ ]:
# Deliberate error. WHERE cannot see an aggregate.
run("""
    SELECT weathersit, SUM(cnt) AS total
    FROM day
    WHERE SUM(cnt) > 1000000
    GROUP BY weathersit
""")

In [ ]:
# The fix: HAVING filters groups, after they have been formed.
show("""
    SELECT weathersit, SUM(cnt) AS total
    FROM day
    GROUP BY weathersit
    HAVING SUM(cnt) > 1000000
""")

> **The rule in one line:** `WHERE` filters **rows** before grouping;
> `HAVING` filters **groups** after. If your condition mentions
> `SUM`, `COUNT`, `AVG`, `MIN` or `MAX`, it belongs in `HAVING`.

You can use both in one query, and often should — `WHERE` first to
discard rows cheaply, then `HAVING` on what survives.

In [ ]:
show("""
    SELECT weathersit,
           COUNT(*)   AS days,
           SUM(cnt)   AS total
    FROM day
    WHERE yr = 1                    -- rows: 2012 only
    GROUP BY weathersit
    HAVING COUNT(*) > 10            -- groups: at least 11 days
    ORDER BY total DESC
""")

---

## 9. Window functions  *(slide 31)*

Slide 31 is the most advanced idea in the deck and it describes it
well:

> *"Operates on a set of rows and return a value for each row. Like
> aggregation, but performed in relation to current row."*

The difference from `GROUP BY` in one sentence: **`GROUP BY` collapses
many rows into one; a window function keeps every row and adds a column
computed from its neighbours.**

The slide's example is a running total, and it even names our dataset:

```sql
SELECT duration, SUM(duration)
OVER (ORDER BY start_time) AS running_total
FROM bikeshare
```

Our bikeshare table has `dteday` and `cnt` rather than `start_time` and
`duration`, so here is the same query in our column names.

In [ ]:
show("""
    SELECT dteday,
           cnt,
           SUM(cnt) OVER (ORDER BY dteday) AS running_total
    FROM day
    ORDER BY dteday
    LIMIT 10
""")

Check it by hand: row 2's `running_total` is row 1's `cnt` plus row 2's
`cnt`. Every row keeps its own value **and** gains the total so far.

### The subtlety slide 31 skates over

The slide says the running total is *"sum of all previous values of
duration"*. It is actually the sum of all preceding rows **and the
current row** — which the output above shows.

But there is a second, sharper detail. `OVER (ORDER BY x)` with no
frame clause defaults to:

```sql
RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

**`RANGE`, not `ROWS`.** With `RANGE`, all rows that *tie* on the
`ORDER BY` value are treated as one unit — every tied row gets the same
answer, including all its peers. With `ROWS`, each row stops at itself.

On `day` there are no ties, so it makes no difference. On the hourly
table grouped by month there are plenty. Watch them disagree.

> **Predict first.** Two running totals over the same rows, one using RANGE and one using ROWS, ordered by a column with ties. Will the first row's two values match?
>
> Put your answer in the chat before we run it.

In [ ]:
show("""
    SELECT mnth,
           cnt,
           SUM(cnt) OVER (ORDER BY mnth
                          RANGE BETWEEN UNBOUNDED PRECEDING
                                    AND CURRENT ROW) AS range_total,
           SUM(cnt) OVER (ORDER BY mnth
                          ROWS  BETWEEN UNBOUNDED PRECEDING
                                    AND CURRENT ROW) AS rows_total
    FROM day
    WHERE yr = 0 AND mnth <= 2
    ORDER BY mnth
    LIMIT 8
""")

The two columns disagree from the first row. `range_total` has already
summed **every January day** on row 1, because every January row ties on
`mnth = 1` and RANGE treats ties as one unit. `rows_total` climbs one
row at a time.

Neither is wrong. They answer different questions, and the *default* is
`RANGE`. If you want a row-by-row running total, **say `ROWS`.**

### PARTITION BY: restart the window

`PARTITION BY` is to window functions what `GROUP BY` is to aggregates —
it splits the rows into independent groups. The running total restarts
in each one.

In [ ]:
show("""
    SELECT yr, mnth, SUM(cnt) AS monthly,
           SUM(SUM(cnt)) OVER (PARTITION BY yr
                               ORDER BY mnth
                               ROWS BETWEEN UNBOUNDED PRECEDING
                                        AND CURRENT ROW) AS ytd
    FROM day
    GROUP BY yr, mnth
    ORDER BY yr, mnth
""")

Read the `ytd` column: it climbs through 2011 (`yr = 0`), then
**resets** at the start of 2012 (`yr = 1`). That is `PARTITION BY yr`.

Note also `SUM(SUM(cnt))` — the inner `SUM` is the aggregate that builds
monthly totals, the outer one is the window function running over those
results. Aggregates run first, windows run over their output.

### Three window functions worth knowing

`ROW_NUMBER`, `RANK` and `DENSE_RANK` differ only in how they handle
ties, and the difference is worth seeing once.

In [ ]:
# The three only differ when there is a TIE, so we need tied values.
# The bikeshare weather counts are all distinct, which would prove nothing.
execute("CREATE TABLE contest (player TEXT, points INTEGER)")
db.executemany("INSERT INTO contest VALUES (?, ?)",
               [("Ada", 90), ("Bo", 85), ("Cy", 85), ("Di", 70)])
db.commit()

ranked = show("""
    SELECT player,
           points,
           ROW_NUMBER() OVER (ORDER BY points DESC) AS row_number,
           RANK()       OVER (ORDER BY points DESC) AS rank,
           DENSE_RANK() OVER (ORDER BY points DESC) AS dense_rank
    FROM contest
""")
print(ranked)
print()
print("Bo and Cy are tied on 85. Read across their two rows.")

# The behaviour claimed in the table below, checked.
assert list(ranked["row_number"]) == [1, 2, 3, 4]
assert list(ranked["rank"]) == [1, 2, 2, 4]
assert list(ranked["dense_rank"]) == [1, 2, 2, 3]
print()
print("ROW_NUMBER 1,2,3,4  |  RANK 1,2,2,4  |  DENSE_RANK 1,2,2,3")

| Function | On a tie | After a tie |
|---|---|---|
| `ROW_NUMBER` | breaks it arbitrarily — 1, 2, 3, 4 | continues |
| `RANK` | gives the same rank — 1, 2, 2, 4 | **skips** |
| `DENSE_RANK` | gives the same rank — 1, 2, 2, 3 | does not skip |

"Top 3 by sales" is ambiguous until you have chosen one of these.

---

## 10. SQL from Python  *(slides 33–37)*

Slide 34 draws the right distinction:

| | Embedded | External |
|---|---|---|
| **What** | the engine is a library inside your process | a server your program connects to over a network |
| **Example** | SQLite | PostgreSQL, SQL Server, MySQL |
| **Driver** | `sqlite3` (standard library) | `pyodbc`, `psycopg2`, `pymysql` |
| **Use when** | self-contained app, prototype, local analysis | many clients share one database |

The deck's own description of embedded databases needs one correction:
slide 34 says an embedded SQL RDBMS *"is emulated by a library"*.
**SQLite does not emulate anything** — it is a complete, ACID-compliant
SQL engine that happens to run in your process instead of a server.
Nothing about it is a simulation.

### The one thing this section really has to teach

Slide 37 builds a query with a Python string. The deck never mentions
what that costs, and it is the most exploited mistake in the history of
web software. So before anything else:

In [ ]:
execute("CREATE TABLE users (username TEXT, role TEXT)")
db.executemany("INSERT INTO users VALUES (?, ?)",
               [("alice", "admin"), ("bob", "staff"), ("carol", "staff")])
db.commit()


def lookup_unsafe(username):
    """Builds SQL by pasting text together. NEVER do this."""
    sql = "SELECT * FROM users WHERE username = '" + username + "'"
    print("SQL sent:", sql)
    return run(sql)


print("Normal use, works fine:")
print(lookup_unsafe("alice"))

> **Predict first.** Now we pass the username `' OR '1'='1`. Look at the SQL that gets built. How many rows come back?
>
> Put your answer in the chat before we run it.

In [ ]:
print("Hostile input:")
print(lookup_unsafe("' OR '1'='1"))
print()
print("The quote closed the string, and OR '1'='1' is always true,")
print("so the WHERE clause matched every row. The lookup returned the")
print("entire user table - including alice, the admin.")

In [ ]:
def lookup_safe(username):
    """Uses a parameter placeholder. The value is never parsed as SQL."""
    return run("SELECT * FROM users WHERE username = ?", (username,))


print("Same hostile input, parameterised:")
result = lookup_safe("' OR '1'='1")
print(result)
print()
print("Zero rows. The database looked for a user whose name is")
print("literally the characters  ' OR '1'='1  , found nobody,")
print("and returned nothing. The input was data, never instructions.")

assert len(result) == 0, "parameterised query must not match anything"
assert len(lookup_safe("alice")) == 1, "normal lookups must still work"
print()
print("Both checks passed.")

> **The rule, and it has no exceptions:** values go in as **parameters**
> (`?` in sqlite3, `%s` in psycopg2 and pymysql), never by building the
> string. The database then treats them as *data* and never as
> *instructions*.
>
> This is not only about attackers. The same bug fires on a customer
> genuinely named **O'Brien** — that apostrophe breaks the query exactly
> the same way.

### Slide 36 will not run

Slide 36's example ends with `print row.table_name`. That is Python 2.
Python 3 made `print` a function. We can prove this without pyodbc
installed, because it is a syntax error — Python refuses to compile it.

In [ ]:
slide_36 = '''
import pyodbc
cnxn = pyodbc.connect("DSN=MSSQL-PYTHON")
cursor = cnxn.cursor()
cursor.tables()
rows = cursor.fetchall()
for row in rows:
    print row.table_name
'''

try:
    compile(slide_36, "slide_36", "exec")
    print("compiled - unexpected")
except SyntaxError as problem:
    print("SyntaxError:", problem.msg)
    print("  line", problem.lineno, ":", problem.text.strip())
    print()
    print("Fix: print(row.table_name)")

### Slide 37 is worth a bug hunt

Slide 37 prints fourteen lines of pyodbc. We cannot run it — pyodbc is
not installed and there is no SQL Server here — but Python's own parser
will confirm the first defect for us, and the rest are readable from
the page.

Here is the slide's code, transcribed exactly.

> **Predict first.** Six things are wrong with the code below. See how many you can name before running the next cell.
>
> Put your answer in the chat before we run it.

In [ ]:
slide_37 = '''
import pyodbc
import pandas.io.sql as psql
import pandas as pd

cxnstr = "Server=myServerAddress;Database=myDB;User Id=myUsername;Password=myPass;"
cxn = pyodbc.connect(cxnstr)
cursor = cnxn.cursor()
cursor.execute("""SELECT ID, FirstName, LastName FROM mytable""")
rows = cursor.fetchone()
objects_list = []
    for row in rows:
        d = collections.OrderedDict()
        d['UserID'] = row.ID
        d['FirstName'] = row.FirstName
        d['LastName'] = row.LastName
cxn.close()
'''

try:
    compile(slide_37, "slide_37", "exec")
    print("compiled - unexpected")
except (SyntaxError, IndentationError) as problem:
    print(type(problem).__name__, ":", problem.msg)
    print("  line", problem.lineno, ":", repr(problem.text))

**The six defects.**

1. **`IndentationError`** — `for row in rows:` is indented under
   `objects_list = []`, which is not a block opener. Python cannot even
   parse the file. That is what the cell above just proved.
2. **`cxn` against `cnxn`** — the connection is bound to `cxn`, then
   `cnxn.cursor()` is called. `NameError` on the very next line after
   the indentation is fixed.
3. **`fetchone()` where `fetchall()` was meant** — `fetchone()` returns
   **one row**, not a list of rows. `for row in rows` then iterates over
   that row's *columns*, and `row.ID` fails because a column value is a
   string, not a row object.
4. **`collections` is never imported** — `collections.OrderedDict()`
   raises `NameError`.
5. **`objects_list` is never appended to** — the loop builds `d` and
   throws it away on the next iteration. The list is empty at the end.
   This is the dangerous kind of bug: no error, just no results.
6. **The password is hardcoded in the source.** It would go into version
   control and stay in the history forever. Use an environment variable
   or a secrets manager.

Two more worth noting: `psql` and `pd` are imported and never used, and
the cursor is never closed.

### How it should look

Here is the same job, written properly, in SQLite so it actually runs.

In [ ]:
# The connection closes itself even if something raises inside the block.
with sqlite3.connect(":memory:") as demo:
    demo.execute("CREATE TABLE mytable (ID INTEGER, FirstName TEXT, LastName TEXT)")
    demo.executemany("INSERT INTO mytable VALUES (?, ?, ?)",
                     [(1, "Ada", "Lovelace"), (2, "Alan", "Turing")])

    cursor = demo.execute("SELECT ID, FirstName, LastName FROM mytable")

    objects_list = []
    for row in cursor.fetchall():          # fetchall, not fetchone
        record = {                         # a plain dict; ordered since 3.7
            "UserID": row[0],
            "FirstName": row[1],
            "LastName": row[2],
        }
        objects_list.append(record)        # actually keep it

    print("collected", len(objects_list), "records:")
    for record in objects_list:
        print("  ", record)

assert len(objects_list) == 2, "both rows should have been collected"

### And the shortest version, which is what you will actually write

For analysis work you rarely want a list of dicts. You want a
DataFrame, and pandas does the loop for you.

In [ ]:
frame = pd.read_sql_query("""
    SELECT weathersit,
           COUNT(*) AS days,
           SUM(cnt) AS total_rides
    FROM day
    GROUP BY weathersit
    ORDER BY total_rides DESC
""", db)

print(type(frame))
print(frame)
print()
print("From here it is the pandas you already know:")
print("share of all rides, by weather:")
print((frame["total_rides"] / frame["total_rides"].sum()).round(4).to_string())

> **Where to draw the line between SQL and pandas.** Do the *filtering,
> joining and aggregating* in SQL, close to the data, so that only the
> rows you need cross into Python. Do the *modelling and plotting* in
> pandas. Pulling ten million rows into a DataFrame in order to filter
> them down to two hundred is the mistake to avoid.

Slide 35's example, for completeness, is valid Python and valid SQL —
but it is **incomplete**. It builds `sql_command` and never runs it:
there is no `connection.execute(sql_command)`, no `commit()`, no
`close()`. A student copying that slide gets no table and no error.

---

## 11. NoSQL  *(slides 41–58)*

**A note on how this section is written.** MongoDB and Neo4j are not
installed here and are not reachable offline, so this notebook will not
print invented output for them. Everything below is either conceptual or
runs in SQLite. Where the deck shows Mongo or Cypher code, we read it
rather than pretend to execute it. Slides 52 and 58 mark both labs as
optional homework, which is the right place to actually run them.

### What "NoSQL" means

Slide 42 defines it as *"storage and retrieval of data that is modelled
by means other than tabular relations"*, and notes it is *"not a new
idea (1960+)"* — which is true and worth saying. Pre-relational
databases were hierarchical and network-shaped. The relational model
won, and then the web's scale brought the alternatives back.

The name is a poor one. It is usually read today as **"not only SQL"**,
which slide 60 hints at when it asks *"why has SQL infiltrated the NoSQL
paradigm?"* The answer is that SQL turned out to be the good part —
many NoSQL systems have since bolted on a SQL-like query language.

### The four types  *(slide 45)*

| Type | Model | Examples |
|---|---|---|
| **Key-value** | a dictionary, at scale | Redis, DynamoDB |
| **Document** | JSON-ish records, no fixed schema | MongoDB, CouchDB |
| **Wide column** | rows with differing sparse columns | Cassandra, HBase |
| **Graph** | nodes and edges as first-class things | Neo4j |

### One claim on slide 44 to push back on

Slide 44 lists as a disadvantage: *"Much slower than RDBMS."*

**As a blanket statement that is not true**, and it is worth correcting
because it hides the real trade-off. A key-value store doing a
single-key lookup is typically *faster* than a relational database doing
the same thing — that is what it is built for. NoSQL systems are
generally slower at the things they *did not* optimise for: multi-table
joins, ad-hoc queries, and transactions spanning many records.

The honest version: **NoSQL systems trade generality for a chosen access
pattern.** Fast on the path they were designed for, poor off it. A
relational database is more uniformly good and rarely the fastest at any
one thing.

The rest of slide 44 is sound, and its first bullet is the important
one: *"many don't support true ACID transactions... application code is
obliged to try to manage concurrency issues."* After §3, you know
exactly how much work that sentence is describing.

### Document databases  *(slides 46–51)*

Slide 47's point is that records need not all have the same fields.
Slide 50 shows three MongoDB documents, and the third nests a name
inside a name:

```javascript
{ name: "sue",  age: 26, status: "A", groups: [ "news", "sports" ] }
{ name: "fred",          status: "A", groups: [ "sports", "hobbies", "cars" ] }
{ name: { first: "fred", last: "bloggs" }, status: "A", groups: [ ... ] }
```

Notice what is happening across those three: `age` is missing from the
second, and in the third `name` changes from a string to an object.
**Both are legal.** That is the flexibility, and it is also the cost —
your application code must now handle every shape that has ever been
written.

Slide 49 is careful and correct: document databases were traditionally
ACID **only at the level of a single document**, and MongoDB 4.0
introduced multi-document transactions.

### You can see the shape without MongoDB

SQLite has JSON support built in, so we can store genuinely
document-shaped records and query inside them. This is not MongoDB, and
the differences are real — but the *idea* of querying into a nested
document is exactly the same.

In [ ]:
import json

execute("CREATE TABLE students (id INTEGER PRIMARY KEY, doc TEXT)")

# Three documents with deliberately different shapes - slide 47's point.
documents = [
    {"name": "sue", "age": 26, "status": "A",
     "groups": ["news", "sports"],
     "scores": [{"type": "homework", "score": 78},
                {"type": "exam", "score": 91}]},

    {"name": "fred", "status": "A",                     # no age
     "groups": ["sports", "hobbies", "cars"],
     "scores": [{"type": "homework", "score": 64},
                {"type": "exam", "score": 55}]},

    {"name": {"first": "fred", "last": "bloggs"},       # name is an object
     "status": "B", "groups": ["sports"],
     "scores": [{"type": "homework", "score": 88}]},
]

for i, document in enumerate(documents, start=1):
    execute("INSERT INTO students VALUES (?, ?)", (i, json.dumps(document)))

print(run("SELECT id, doc FROM students").to_string())

In [ ]:
# Query INTO the documents, without a schema ever being declared.
show("""
    SELECT id,
           json_extract(doc, '$.name')   AS name,
           json_extract(doc, '$.age')    AS age,
           json_extract(doc, '$.status') AS status,
           json_array_length(json_extract(doc, '$.groups')) AS n_groups
    FROM students
""")

Read the `name` and `age` columns. `age` came back **empty** for the two
students who never had one — no error, no missing column, just absent.
(pandas shows it as `NaN`; inside SQLite it is NULL.) And row 3's `name`
came back as raw JSON, `{"first":"fred","last":"bloggs"}`, because it is
an object where the others held a string — so a query that expected a
name to be text now has to cope with a dictionary.

**That is document-database life in one table.** Enormous flexibility on
write; all the burden moved onto whoever reads it.

Slide 51's MongoDB example loops through documents in Python to find a
minimum homework score. Here is the same question answered inside the
database.

In [ ]:
show("""
    SELECT json_extract(s.doc, '$.name') AS name,
           MIN(json_extract(score.value, '$.score')) AS min_homework
    FROM students s,
         json_each(json_extract(s.doc, '$.scores')) AS score
    WHERE json_extract(score.value, '$.type') = 'homework'
    GROUP BY s.id
""")

`json_each` expands the nested `scores` array into rows, which we then
group. Slide 51 does the same job with two nested Python loops and a
sentinel value (`minhs = 101`).

Both are valid. The difference is where the work happens: slide 51 pulls
every document across to Python and loops over it there; this pushes the
work into the database and brings back only the three answers. On three
students it does not matter. On three million it is the whole game.

### Graph databases  *(slides 53–58)*

Slide 54's model: **nodes** are entities, **edges** are relationships,
and both can carry properties. Slide 57's Cypher example is correct —
here it is:

```cypher
MATCH (john {name: 'John'})-[:friend]->()-[:friend]->(fof)
RETURN john.name, fof.name
```

Read the pattern as a picture: start at John, follow a `friend` edge to
*someone* (unnamed, `()`), follow another `friend` edge from them to
`fof`. Given the slide's graph — John→Joe, John→Sara, Joe→Steve,
Sara→Maria — the answer is Steve and Maria, which is what the slide's
output shows.

**Why a graph database rather than joins?** You can absolutely store
this relationally: a `people` table and a `friendships` table. Here it
is.

In [ ]:
execute("CREATE TABLE people (id INTEGER PRIMARY KEY, name TEXT)")
execute("""CREATE TABLE friendships (
                from_id INTEGER REFERENCES people(id),
                to_id   INTEGER REFERENCES people(id))""")

db.executemany("INSERT INTO people VALUES (?, ?)",
               [(1, "John"), (2, "Joe"), (3, "Sara"),
                (4, "Steve"), (5, "Maria")])
db.executemany("INSERT INTO friendships VALUES (?, ?)",
               [(1, 2), (1, 3), (2, 4), (3, 5)])
db.commit()

# Friends of friends of John: the same question, as a two-step join.
show("""
    SELECT john.name AS john_name, fof.name AS fof_name
    FROM people john
    JOIN friendships f1  ON john.id = f1.from_id
    JOIN friendships f2  ON f1.to_id = f2.from_id
    JOIN people fof      ON f2.to_id = fof.id
    WHERE john.name = 'John'
""")

Same answer: Steve and Maria.

So why does Neo4j exist? **Count the joins.** Friends-of-friends needed
two. Friends-of-friends-of-friends needs three. Depth *n* needs *n*
joins, and you must write them out — you cannot ask a relational
database for "everyone connected to John by any path" in standard SQL
without recursion, and the cost grows badly.

In Cypher, depth is a number you change:

```cypher
MATCH (john {name:'John'})-[:friend*1..5]->(reachable)
```

**That is the whole argument for graph databases:** when the
*relationships* are the thing you query — social networks, fraud rings,
recommendations, supply chains — a database built on edges beats one
that has to rediscover them with a join every time.

Slide 55's claim that Neo4j is *"most popular graph database"* was true
when written and matches independent database popularity rankings, but
it is a market-share claim with a date on it, not a permanent fact.

---

## 12. Normalisation  *(appendix, slides 63–65)*

Three normal forms, all from Codd. The deck's examples are correct;
here they are as tables you can query.

### First normal form  *(slide 63)*

> *The domain of each attribute contains only atomic (indivisible)
> values, and the value of each attribute contains only a single value
> from that domain.*

In plain words: **one value per cell. No lists.**

The slide's "before" table stores
`"555-861-2025, 192-122-1111"` in one telephone cell.

In [ ]:
# The BEFORE table - violates 1NF.
execute("""CREATE TABLE customer_bad (
    CustomerID INTEGER, FirstName TEXT, Surname TEXT, Telephone TEXT)""")

db.executemany("INSERT INTO customer_bad VALUES (?, ?, ?, ?)", [
    (123, "Pooja", "Singh", "555-861-2025, 192-122-1111"),
    (456, "San", "Zhang", "(555) 403-1659 Ext. 53; 182-929-2929"),
    (789, "John", "Doe", "555-808-9633"),
])
db.commit()

print(run("SELECT * FROM customer_bad").to_string())

> **Predict first.** Using this table, how would you answer: 'how many phone numbers does customer 456 have?' or 'find the customer with number 192-122-1111'?
>
> Put your answer in the chat before we run it.

In [ ]:
# The obvious query is wrong, and it fails silently.
print("Looking for the customer whose number is 192-122-1111:")
print(run("SELECT * FROM customer_bad WHERE Telephone = '192-122-1111'"))
print("-> no rows. The number IS in the table, inside a longer string.")
print()

print("LIKE finds it, but now look what else it finds:")
print(run("SELECT CustomerID, Telephone FROM customer_bad "
          "WHERE Telephone LIKE '%555%'").to_string())
print()
print("Two customers, because '555' appears inside other numbers too.")
print("And counting numbers per customer means parsing text -")
print("except one row separates with a comma and another with a semicolon.")

In [ ]:
# The AFTER table - 1NF. One number per row.
execute("""CREATE TABLE customer_good (
    CustomerID INTEGER, FirstName TEXT, Surname TEXT, Telephone TEXT)""")

db.executemany("INSERT INTO customer_good VALUES (?, ?, ?, ?)", [
    (123, "Pooja", "Singh", "555-861-2025"),
    (123, "Pooja", "Singh", "192-122-1111"),
    (456, "San", "Zhang", "182-929-2929"),
    (456, "San", "Zhang", "(555) 403-1659 Ext. 53"),
    (789, "John", "Doe", "555-808-9633"),
])
db.commit()

print("Now both questions are ordinary SQL:")
print()
print(show("SELECT * FROM customer_good WHERE Telephone = '192-122-1111'"))
print()
print(show("""
    SELECT CustomerID, COUNT(*) AS phone_numbers
    FROM customer_good
    GROUP BY CustomerID
"""))

**One thing the slide does not mention, and you should notice.** In the
1NF table, `CustomerID` is no longer unique — 123 appears twice. So it
can no longer be the primary key on its own. The key is now the
*combination* `(CustomerID, Telephone)`, called a **composite key**.

Strictly, this table is now carrying a different problem: Pooja's name
is stored twice. Fixing that is what 2NF is about.

### Second normal form  *(slide 64)*

> *In 1NF, and no non-prime attribute is dependent on any proper subset
> of any candidate key.*

That sentence is precise and nearly unreadable. The plain version:
**if your key has two parts, every other column must depend on
both parts, not just one.**

The slide's example is electric toothbrushes, keyed on
`(Manufacturer, Model)`. `Manufacturer Country` depends only on
`Manufacturer` — half the key. So it is split into its own table.

In [ ]:
execute("""CREATE TABLE toothbrush_bad (
    Manufacturer TEXT, Model TEXT,
    ModelFullName TEXT, ManufacturerCountry TEXT)""")

db.executemany("INSERT INTO toothbrush_bad VALUES (?, ?, ?, ?)", [
    ("Forte", "X-Prime", "Forte X-Prime", "Italy"),
    ("Forte", "Ultraclean", "Forte Ultraclean", "Italy"),
    ("Dent-o-Fresh", "EZbrush", "Dent-o-Fresh EZbrush", "USA"),
    ("Kobayashi", "ST-60", "Kobayashi ST-60", "Japan"),
    ("Hoch", "Toothmaster", "Hoch Toothmaster", "Germany"),
    ("Hoch", "X-Prime", "Hoch X-Prime", "Germany"),
])
db.commit()

print(run("SELECT * FROM toothbrush_bad").to_string())
print()
print(show("""
    SELECT Manufacturer,
           COUNT(*) AS rows_repeating_the_country,
           ManufacturerCountry
    FROM toothbrush_bad
    GROUP BY Manufacturer, ManufacturerCountry
    HAVING COUNT(*) > 1
"""))
print("Italy is stored twice, Germany twice. Correct one and miss the")
print("other, and the database now disagrees with itself.")

In [ ]:
# 2NF: split the fact that depends on only part of the key.
execute("CREATE TABLE manufacturers (Manufacturer TEXT PRIMARY KEY, Country TEXT)")
execute("""CREATE TABLE toothbrush_models (
    Manufacturer TEXT, Model TEXT, ModelFullName TEXT,
    PRIMARY KEY (Manufacturer, Model))""")

db.executemany("INSERT INTO manufacturers VALUES (?, ?)",
               [("Forte", "Italy"), ("Dent-o-Fresh", "USA"),
                ("Kobayashi", "Japan"), ("Hoch", "Germany")])
db.executemany("INSERT INTO toothbrush_models VALUES (?, ?, ?)", [
    ("Forte", "X-Prime", "Forte X-Prime"),
    ("Forte", "Ultraclean", "Forte Ultraclean"),
    ("Dent-o-Fresh", "EZbrush", "Dent-o-Fresh EZbrush"),
    ("Kobayashi", "ST-60", "Kobayashi ST-60"),
    ("Hoch", "Toothmaster", "Hoch Toothmaster"),
    ("Hoch", "X-Prime", "Hoch X-Prime"),
])
db.commit()

print("Each country is now stored exactly once:")
print(run("SELECT * FROM manufacturers").to_string())
print()
print("And the join puts it back whenever you want it:")
print(show("""
    SELECT m.Model, m.ModelFullName, f.Country
    FROM toothbrush_models m
    JOIN manufacturers f ON m.Manufacturer = f.Manufacturer
    ORDER BY f.Country, m.Model
"""))

### Third normal form  *(slide 65)*

> *In 2NF, and every non-prime attribute is non-transitively dependent
> on every key.*

Plain version: **no column may depend on another non-key column.**

The slide's example: a tournament winners table keyed on
`(Tournament, Year)`, carrying `Winner` and `Winner Date of Birth`.
The date of birth does not depend on the tournament at all — it depends
on the *winner*, which is itself not a key. That is the transitive step,
and it is why the fix is a separate table of winners.

In [ ]:
execute("""CREATE TABLE tournament_bad (
    Tournament TEXT, Year INTEGER, Winner TEXT, WinnerDOB TEXT)""")

db.executemany("INSERT INTO tournament_bad VALUES (?, ?, ?, ?)", [
    ("Indiana Invitational", 1998, "Al Fredrickson", "21 July 1975"),
    ("Cleveland Open", 1999, "Bob Albertson", "28 September 1968"),
    ("Des Moines Masters", 1999, "Al Fredrickson", "21 July 1975"),
    ("Indiana Invitational", 1999, "Chip Masterson", "14 March 1977"),
])
db.commit()

print(run("SELECT * FROM tournament_bad").to_string())
print()
print("Al Fredrickson's date of birth appears twice. Nothing in the")
print("schema stops the two copies from disagreeing. Watch:")

execute("""UPDATE tournament_bad SET WinnerDOB = '22 July 1975'
           WHERE Tournament = 'Des Moines Masters'""")

print()
print(show("""
    SELECT Winner, COUNT(DISTINCT WinnerDOB) AS number_of_birthdays
    FROM tournament_bad
    GROUP BY Winner
    HAVING COUNT(DISTINCT WinnerDOB) > 1
"""))
print("Al Fredrickson now has two dates of birth. The database cannot")
print("tell you which is right, because it does not know they must match.")

In [ ]:
# 3NF: the winner's date of birth belongs to the winner, not the tournament.
execute("CREATE TABLE winners (Winner TEXT PRIMARY KEY, DOB TEXT)")
execute("""CREATE TABLE tournament_winners (
    Tournament TEXT, Year INTEGER, Winner TEXT REFERENCES winners(Winner),
    PRIMARY KEY (Tournament, Year))""")

db.executemany("INSERT INTO winners VALUES (?, ?)", [
    ("Chip Masterson", "14 March 1977"),
    ("Al Fredrickson", "21 July 1975"),
    ("Bob Albertson", "28 September 1968"),
])
db.executemany("INSERT INTO tournament_winners VALUES (?, ?, ?)", [
    ("Indiana Invitational", 1998, "Al Fredrickson"),
    ("Cleveland Open", 1999, "Bob Albertson"),
    ("Des Moines Masters", 1999, "Al Fredrickson"),
    ("Indiana Invitational", 1999, "Chip Masterson"),
])
db.commit()

birthdays = run("""
    SELECT Winner, COUNT(DISTINCT DOB) AS number_of_birthdays
    FROM winners GROUP BY Winner
""")
print(birthdays)
print()
print("Each winner now has exactly one date of birth, and the schema")
print("makes any other state impossible - PRIMARY KEY (Winner) sees to it.")

assert birthdays["number_of_birthdays"].max() == 1

### When to stop normalising

The three forms in one line each:

| Form | Rule |
|---|---|
| **1NF** | one value per cell |
| **2NF** | 1NF, and no column depends on only *part* of a composite key |
| **3NF** | 2NF, and no column depends on another non-key column |

**Normalisation is not automatically good.** Every split you make is a
join you must write later. Transactional systems — the ones taking
orders, moving money — normalise hard, because their enemy is
inconsistency. Analytical systems and data warehouses often
**deliberately denormalise**, because their enemy is join cost and their
data is written once and read a thousand times.

Slide 6's three columns are exactly this split: **Operations**
(transactions, normalise) against **Data Warehouse** (reporting and
analytics, often denormalised).

The judgement, then: **normalise until it hurts, denormalise until it
works.**

---

## The checklist

Before you trust a query you have written:

1. **Did I mean `COUNT(*)` or `COUNT(column)`?** They differ whenever
   NULL is possible.
2. **Is my condition on rows or on groups?** `WHERE` before, `HAVING`
   after.
3. **Does every join have an `ON`?** *n* tables need *n − 1* conditions.
   A missing one is a silent cross join.
4. **Did I mean to lose the unmatched rows?** `INNER` drops them.
   `LEFT` keeps them.
5. **Are NULLs doing something I did not intend?** `AVG` skips them,
   and `NULL = NULL` is not true — use `IS NULL`.
6. **Are values parameterised?** `?`, never string concatenation. Every
   time.
7. **Is this filtering happening in the database or in pandas?** Filter
   before the data crosses over, not after.

## Practice — there is a database waiting for you

Everything above was built in memory and disappears when you close
Python. For practice you want something that persists, that you can open
in a proper database tool, and that has enough rows to be interesting.

**`data/bikeshare.db`** — 2.2 MB, 19,478 rows across 7 tables and a
view. All real data: the Capital Bikeshare rides you already know, a
real 858-station GBFS feed, and the Boston housing table.

**`15-sql-practice.ipynb`** — 21 exercises against it, from `SELECT` to
window functions, each one self-marking. Write your SQL, run the cell,
and it tells you whether you are right without showing you the answer.

You can also open the `.db` with no Python at all — DB Browser for
SQLite, the VS Code SQLite extension, or <https://sqliteonline.com/> in
a browser. **No account, no signup, nothing to install.** Full schema in
`data/bikeshare-db-README.md`.

Two things in that database are worth meeting deliberately, and neither
was invented for teaching — they are simply true of the source data:

- **Weather code 4 never occurs in `rides_daily`**, though it exists in
  the `weather` lookup and does occur hourly. So `INNER JOIN` returns 3
  rows and `LEFT JOIN` returns 4. §7, on real data.
- **54 of 858 stations have a NULL `region_id`.** Real missing data for
  the `IS NULL` lesson in §8.

## Your turn

All against tables already in this notebook.

1. **Busiest and quietest.** For each season, return total rides,
   average rides per day, and the number of days. Order by total rides
   descending. *(§8)*
2. **The NULL trap, deliberately.** Add a row to `survey` with a NULL
   score, then write two queries that give different "average" answers,
   and say in one sentence which question each answers. *(§8)*
3. **Find the orphans.** Insert a recording with a `GenreID` that does
   not exist, with foreign keys OFF. Then write the `LEFT JOIN` that
   finds it. Then turn foreign keys ON and show the insert being
   refused. *(§5)*
4. **A join you must get right.** List every artist together with their
   number of recordings — **including artists with none.** Getting a
   zero to appear is the whole exercise. *(§7)*
5. **Running total, correctly.** Produce a day-by-day running total of
   rides for 2012 only, and explain in one line why you used `ROWS`
   rather than the default. *(§9)*

## Stretch

1. **Break a cross join on purpose.** Cross-join `hour` with itself,
   `LIMIT 5`. Then work out — do not run — how many rows the unlimited
   query would return. Compare with the number of seconds since the
   universe began.
2. **Index economics.** Time an insert of 10,000 rows into a table with
   no index, then into the same table with three indexes. Measure the
   cost slide 18's last bullet is warning about.
3. **Rebuild §11 relationally.** Take the three JSON student documents
   and design a normalised schema for them. Then say which you would
   rather query, and which you would rather have to change next month.
4. **Recursive SQL.** Look up `WITH RECURSIVE` and use it to find
   everyone reachable from John at *any* depth in `friendships`. This is
   the query that graph databases exist to make easy — write it once and
   you will understand why.

## Where to go deeper

Checked 9 September 2026.

- **SQLite documentation** — <https://www.sqlite.org/docs.html>. The
  pages on [type affinity](https://www.sqlite.org/datatype3.html) and
  [query planning](https://www.sqlite.org/queryplanner.html) are the two
  that explain the surprises in §5 and §6.
- **ThoughtSpot's SQL tutorial** (slides 24 and 32) —
  <https://www.thoughtspot.com/sql-tutorial/introduction-to-sql>
- **`sqlite3` module docs** —
  <https://docs.python.org/3/library/sqlite3.html>. Read the section on
  placeholders; it is §10 in three paragraphs.
- **Use The Index, Luke** — <https://use-the-index-luke.com/>. Free, and
  the best explanation of indexes anywhere.
- **Codd, E. F. (1970)**, "A Relational Model of Data for Large Shared
  Data Banks", *Communications of the ACM* 13(6), 377–387. The paper
  the whole of §2 and §12 comes from.

---

*Data Science & AI — Module 3 Part 1. Covers `Module_3_Part_1.pdf`,
slides 4–37, 41–58 and the normalisation appendix 63–65. Official labs:
IOD Lab 3.1.1 and 3.1.2 (SQL), 3.1.3 (SQL in Python), 3.1.4 (MongoDB)
and 3.1.5 (Neo4j) — the last two optional homework.*